# Support Vector Machines (SVM)
### Clasificación con Máquinas de Vectores de Soporte
### Aplicado al dataset *Breast Cancer* (scikit-learn)

Notebook educativo/demostrativo para Google Colab.


## 1. Descripción

**Support Vector Machines (SVM)** es un modelo que sirve para **separar datos en categorías** (por ejemplo: "tumor benigno" vs "tumor maligno").

La idea, en palabras simples, es:

- Imagina los datos como puntos en un plano (o en un espacio de más dimensiones).
- SVM busca la **línea (o superficie) que mejor separa las dos categorías**.
- No busca cualquier línea que separe los puntos, sino la que deja el **mayor espacio posible** (margen) entre las dos categorías. Esto ayuda a que el modelo generalice mejor con datos nuevos.
- Los puntos que están **justo en el borde** de ese margen son los más importantes: se llaman **vectores de soporte**, y son los que realmente "sostienen" la decisión del modelo. El resto de los puntos, más alejados, casi no influyen.
- Cuando los datos **no se pueden separar con una línea recta**, SVM usa un truco llamado **kernel**, que transforma los datos a un espacio donde sí es más fácil separarlos con un plano.

En este notebook aplicamos SVM al **dataset Breast Cancer** de scikit-learn, que contiene mediciones de células extraídas de tumores mamarios, y el objetivo es predecir si un tumor es **benigno** o **maligno**.


## 2. Bibtex y Referencias

**Referencias principales:**

- Cortes, C., & Vapnik, V. (1995). Support-Vector Networks. *Machine Learning*, 20(3), 273-297.
- Vapnik, V. N. (1998). *Statistical Learning Theory*. Wiley.
- Boser, B. E., Guyon, I. M., & Vapnik, V. N. (1992). A training algorithm for optimal margin classifiers. *Proceedings of the 5th Annual Workshop on Computational Learning Theory (COLT)*.
- Wolberg, W. H., Street, W. N., & Mangasarian, O. L. (1995). Breast Cancer Wisconsin (Diagnostic) Data Set. UCI Machine Learning Repository (fuente original del dataset Breast Cancer).

**BibTeX:**

```bibtex
@article{cortes1995support,
  title={Support-vector networks},
  author={Cortes, Corinna and Vapnik, Vladimir},
  journal={Machine Learning},
  volume={20},
  number={3},
  pages={273--297},
  year={1995},
  publisher={Springer}
}

@book{vapnik1998statistical,
  title={Statistical Learning Theory},
  author={Vapnik, Vladimir N.},
  year={1998},
  publisher={Wiley}
}

@inproceedings{boser1992training,
  title={A training algorithm for optimal margin classifiers},
  author={Boser, Bernhard E. and Guyon, Isabelle M. and Vapnik, Vladimir N.},
  booktitle={Proceedings of the fifth annual workshop on Computational learning theory},
  pages={144--152},
  year={1992}
}

@misc{wolberg1995breast,
  title={Breast Cancer Wisconsin (Diagnostic) Data Set},
  author={Wolberg, William H. and Street, W. Nick and Mangasarian, Olvi L.},
  howpublished={UCI Machine Learning Repository},
  year={1995}
}
```


## 3. Tipo de modelo

| Criterio | Clasificación |
|---|---|
| **Tipo general** | Aprendizaje supervisado (clasificación; también existe una versión para regresión, SVR) |
| **Método de aprendizaje** | **Basado en instancias** (*instance-based*). Aunque SVM sí resuelve una optimización durante el entrenamiento, el modelo final se define únicamente en función de un subconjunto de los datos de entrenamiento: los **vectores de soporte**. La predicción de un nuevo punto depende de comparar (mediante el kernel) ese punto contra los vectores de soporte guardados, no contra una fórmula cerrada e independiente de los datos |
| **Por parámetros** | En su forma con kernel (no lineal) se comporta como un modelo **no paramétrico**: el número de vectores de soporte (y por lo tanto la "complejidad" del modelo) depende de los datos, no de un número fijo definido de antemano. En su versión lineal simple, sí puede verse como paramétrica (un vector de pesos $w$ y un sesgo $b$) |
| **Datos de aprendizaje** | Requiere conservar los **vectores de soporte** (un subconjunto de los datos de entrenamiento) para poder predecir nuevos puntos; no descarta por completo los datos después de entrenar |
| **Resultado del aprendizaje** | Un **hiperplano de separación** (frontera de decisión) definido por los vectores de soporte, sus pesos (multiplicadores de Lagrange $\alpha_i$) y el sesgo $b$ |


## 4. Algoritmo de entrenamiento

**Nombre del algoritmo:** *Support Vector Classification* mediante **optimización de margen máximo**, resuelta típicamente con el algoritmo **SMO (Sequential Minimal Optimization)**, propuesto por Platt (1998) y usado internamente por scikit-learn (`libsvm`).

**Idea del entrenamiento:**

1. Se plantea el problema de encontrar el hiperplano $w^T x + b = 0$ que **maximiza el margen** entre las dos clases, es decir, minimizar:
   $$\frac{1}{2}\|w\|^2 + C \sum_{i=1}^{m} \xi_i$$
   sujeto a que cada punto esté correctamente clasificado (permitiendo algo de error $\xi_i$ si los datos no son perfectamente separables).
2. El hiperparámetro $C$ controla el compromiso entre un margen amplio (más tolerante a errores) y clasificar correctamente todos los puntos de entrenamiento (menos tolerante a errores).
3. Este problema se resuelve más fácilmente en su forma **dual**, donde aparecen los multiplicadores de Lagrange $\alpha_i$ y los datos solo intervienen a través de productos internos $x_i \cdot x_j$.
4. Ese producto interno se puede reemplazar por una **función kernel** $K(x_i, x_j)$ (lineal, polinomial, RBF/gaussiano, etc.), lo que permite separar datos que no son linealmente separables en el espacio original, sin transformar explícitamente los datos.
5. El algoritmo **SMO** resuelve esta optimización de forma eficiente, actualizando de a pares de $\alpha_i$ en cada iteración hasta converger.
6. Al final, solo los puntos con $\alpha_i > 0$ (los vectores de soporte) definen la frontera de decisión.


## 5. Supuestos y restricciones

**Supuestos:**
- Existe una frontera (lineal, o no lineal si se usa un kernel) que separa razonablemente bien las clases.
- Las variables predictoras están en una escala comparable (SVM es sensible a la escala, ya que se basan en distancias/productos internos).
- Los datos de entrenamiento son representativos de la población sobre la que se harán predicciones futuras.

**Restricciones / limitaciones:**
- **Sensible a la escala de las variables:** si una variable tiene valores mucho más grandes que otras, puede dominar el cálculo del margen; por eso se recomienda estandarizar los datos antes de entrenar.
- **Costo computacional:** el entrenamiento puede ser lento en datasets muy grandes (la complejidad crece entre $O(m^2)$ y $O(m^3)$ según el kernel usado).
- **Sensible a los hiperparámetros:** la elección de $C$ (regularización) y de los parámetros del kernel (por ejemplo, $\gamma$ en el kernel RBF) afecta mucho el resultado; valores mal elegidos pueden causar sobreajuste o subajuste.
- **Menos directo de interpretar** que un modelo lineal simple, especialmente al usar kernels no lineales.
- **No estima directamente probabilidades**; SVM da una decisión de clase, y aunque existen métodos para aproximar probabilidades (`probability=True` en scikit-learn), esto agrega costo computacional extra.
- Sensible a **datos desbalanceados** (una clase con muchos más ejemplos que la otra) si no se ajustan los pesos de clase.


## 6. Implementación en Python

### Paso 1: Importar librerías

**Qué hacemos:** importamos las librerías necesarias.

**Por qué así:**
- `numpy` y `pandas` para manejar los datos numéricamente y en forma de tabla.
- `matplotlib` para graficar y entender visualmente el modelo.
- `sklearn.datasets` para cargar el dataset Breast Cancer, ya incluido en scikit-learn.
- `sklearn.model_selection` para dividir los datos en entrenamiento y prueba.
- `sklearn.preprocessing` para estandarizar las variables (importante en SVM, como se explicó en los supuestos).
- `sklearn.svm` para usar la implementación de SVM (`SVC`) ya optimizada, en vez de programarla desde cero (SVM requiere resolver un problema de optimización cuadrática, poco práctico de implementar manualmente en un notebook educativo).
- `sklearn.metrics` para evaluar qué tan bien clasifica el modelo.
- Fijamos `np.random.seed(42)` para que los resultados sean reproducibles.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.decomposition import PCA

np.random.seed(42)  # semilla fija -> resultados reproducibles cada vez que se ejecute el notebook


### Paso 2: Cargar el dataset Breast Cancer

**Qué hacemos:** cargamos los datos con `load_breast_cancer(as_frame=True)`.

**Por qué así:**
- `as_frame=True` entrega los datos como DataFrame de pandas, con nombres de columnas legibles.
- El dataset contiene 30 mediciones numéricas de núcleos celulares (radio, textura, perímetro, área, etc.) y una etiqueta binaria: **0 = maligno**, **1 = benigno**.
- Ya viene limpio y sin valores faltantes, por lo que no se necesita un paso previo de limpieza.


In [ ]:
data = load_breast_cancer(as_frame=True)
df = data.frame

print(data.DESCR[:1000])
df.head()


In [ ]:
print("Clases:", dict(zip(data.target_names, [0, 1])))
print("\nDistribucion de clases:")
print(df["target"].value_counts())


### Paso 3: Separar variables predictoras (X) y variable objetivo (y)

**Qué hacemos:** separamos las 30 columnas de mediciones (`X`) de la columna `target` (`y`), que indica si el tumor es benigno o maligno.

**Por qué así:** el modelo necesita, por un lado, las variables que va a usar para predecir, y por otro, la respuesta correcta que va a intentar aprender a predecir.


In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)


### Paso 4: Separar en entrenamiento y prueba

**Qué hacemos:** dividimos los datos en 80% entrenamiento y 20% prueba.

**Por qué así:**
- Igual que en el notebook anterior, necesitamos datos que el modelo **no haya visto** para evaluar si realmente aprendió a distinguir tumores benignos de malignos, y no solo memorizó los datos de entrenamiento.
- `stratify=y` es importante aquí: asegura que la proporción de tumores benignos/malignos sea **la misma** en el conjunto de entrenamiento y en el de prueba. Esto evita que, por azar, el conjunto de prueba quede con muy pocos ejemplos de una clase.
- `random_state=42` para que la división sea siempre la misma (reproducibilidad).


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras")


### Paso 5: Estandarizar las variables

**Qué hacemos:** aplicamos `StandardScaler` para que todas las variables queden con media 0 y desviación estándar 1.

**Por qué así:** como se explicó en los "Supuestos y restricciones", SVM calcula distancias y márgenes entre puntos. Si una variable (por ejemplo, "área") tiene valores mucho más grandes que otra (por ejemplo, "simetría"), dominaría el cálculo del margen sin ser necesariamente más importante. Estandarizar pone a todas las variables en pie de igualdad.

Importante: el `scaler` se **ajusta (fit)** solo con los datos de entrenamiento, y luego se **aplica (transform)** tanto a entrenamiento como a prueba. Esto evita que información del conjunto de prueba "se filtre" al proceso de entrenamiento (fuga de datos).


In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # ajusta y transforma con datos de entrenamiento
X_test_s = scaler.transform(X_test)         # solo transforma, usando lo aprendido del entrenamiento


### Paso 6: Entrenar el modelo SVM (kernel lineal)

**Qué hacemos:** entrenamos un `SVC` con `kernel='linear'`.

**Por qué así:**
- Empezamos con el kernel más simple, **lineal**, porque es el más fácil de interpretar (busca directamente un hiperplano separador) y suele funcionar bien en este dataset, donde las clases tienden a ser bastante separables.
- `C=1.0` es el valor por defecto: un punto intermedio entre un margen amplio (más tolerante a errores de clasificación) y un margen estricto (clasifica bien todos los puntos de entrenamiento, con riesgo de sobreajuste). Más adelante probamos otros valores.
- `random_state=42` para reproducibilidad (algunos solvers internos usan aleatoriedad).


In [ ]:
svm_linear = SVC(kernel="linear", C=1.0, random_state=42)
svm_linear.fit(X_train_s, y_train)

y_pred_linear = svm_linear.predict(X_test_s)

print(f"Accuracy (kernel lineal): {accuracy_score(y_test, y_pred_linear):.4f}")
print(f"Numero de vectores de soporte: {svm_linear.n_support_} (por clase)")
print(f"Total de vectores de soporte: {svm_linear.support_vectors_.shape[0]} de {X_train_s.shape[0]} muestras de entrenamiento")


### Paso 7: Evaluar el modelo con métricas de clasificación

**Qué hacemos:** calculamos la matriz de confusión y el reporte de clasificación (precisión, recall, F1).

**Por qué así:** en un problema médico como este, el **accuracy** solo no es suficiente. Es clave revisar, por ejemplo, cuántos tumores malignos fueron clasificados incorrectamente como benignos (falsos negativos), ya que ese tipo de error es más grave que el contrario. La matriz de confusión y el reporte muestran ese detalle por clase.


In [ ]:
cm = confusion_matrix(y_test, y_pred_linear)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=data.target_names)
disp.plot(cmap="Blues")
plt.title("Matriz de confusion - SVM (kernel lineal)")
plt.show()

print(classification_report(y_test, y_pred_linear, target_names=data.target_names))


### Paso 8: Probar distintos kernels

**Qué hacemos:** entrenamos SVM con varios kernels (`linear`, `poly`, `rbf`, `sigmoid`) y comparamos su accuracy.

**Por qué así:** cada kernel asume una forma distinta de frontera de decisión. Comparar varios permite ver cuál se ajusta mejor a este dataset en particular, en vez de asumir de antemano que uno es el mejor. `gamma='scale'` se deja como valor por defecto recomendado por scikit-learn, que ajusta automáticamente ese parámetro según la varianza de los datos.


In [ ]:
kernels = ["linear", "poly", "rbf", "sigmoid"]
resultados = {}

for k in kernels:
    modelo = SVC(kernel=k, C=1.0, gamma="scale", random_state=42)
    modelo.fit(X_train_s, y_train)
    y_pred = modelo.predict(X_test_s)
    acc = accuracy_score(y_test, y_pred)
    resultados[k] = acc
    print(f"Kernel = {k:8s} -> Accuracy = {acc:.4f} | Vectores de soporte = {modelo.support_vectors_.shape[0]}")

plt.figure(figsize=(7, 4))
plt.bar(resultados.keys(), resultados.values(), color="steelblue")
plt.ylim(0.8, 1.0)
plt.ylabel("Accuracy")
plt.title("Comparacion de kernels en SVM - Breast Cancer")
plt.show()


### Paso 9: Efecto del hiperparámetro C (kernel RBF)

**Qué hacemos:** probamos distintos valores de `C` con el kernel `rbf` y observamos cómo cambia el accuracy y el número de vectores de soporte.

**Por qué así:**
- `C` chico → el modelo prioriza un margen amplio y tolera más errores de clasificación en el entrenamiento (puede generar **subajuste**).
- `C` grande → el modelo prioriza clasificar bien todos los puntos de entrenamiento, con un margen más estrecho (puede generar **sobreajuste**).
- Ver cómo cambia el número de vectores de soporte ayuda a entender esto: con `C` grande, el modelo tiende a "memorizar" más puntos como vectores de soporte.


In [ ]:
C_values = [0.01, 0.1, 1, 10, 100]
acc_por_C = []
n_sv_por_C = []

for c in C_values:
    modelo = SVC(kernel="rbf", C=c, gamma="scale", random_state=42)
    modelo.fit(X_train_s, y_train)
    y_pred = modelo.predict(X_test_s)
    acc_por_C.append(accuracy_score(y_test, y_pred))
    n_sv_por_C.append(modelo.support_vectors_.shape[0])

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.set_xlabel("C (escala log)")
ax1.set_ylabel("Accuracy", color="steelblue")
ax1.plot(C_values, acc_por_C, marker="o", color="steelblue", label="Accuracy")
ax1.set_xscale("log")
ax1.tick_params(axis="y", labelcolor="steelblue")

ax2 = ax1.twinx()
ax2.set_ylabel("N. vectores de soporte", color="darkorange")
ax2.plot(C_values, n_sv_por_C, marker="s", color="darkorange", label="Vectores de soporte")
ax2.tick_params(axis="y", labelcolor="darkorange")

plt.title("Efecto de C en SVM (kernel RBF)")
fig.tight_layout()
plt.show()


### Paso 10 (extra): Visualizar la frontera de decisión en 2D

**Qué hacemos:** reducimos los datos a 2 dimensiones con **PCA** (análisis de componentes principales) y entrenamos un SVM sobre esas 2 dimensiones, solo para poder **graficar** la frontera de decisión.

**Por qué así:** el dataset real tiene 30 variables, imposible de graficar directamente. PCA combina esas 30 variables en 2 "super-variables" que conservan la mayor parte de la información posible, permitiendo visualizar cómo SVM separa las clases. Esto es solo para fines ilustrativos: el modelo real (Pasos 6-9) usa las 30 variables originales, no esta versión reducida.


In [ ]:
pca = PCA(n_components=2, random_state=42)
X_train_2d = pca.fit_transform(X_train_s)

svm_2d = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42)
svm_2d.fit(X_train_2d, y_train)

# Malla de puntos para dibujar la frontera de decision
x_min, x_max = X_train_2d[:, 0].min() - 1, X_train_2d[:, 0].max() + 1
y_min, y_max = X_train_2d[:, 1].min() - 1, X_train_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
Z = svm_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.25, cmap="coolwarm")
scatter = plt.scatter(X_train_2d[:, 0], X_train_2d[:, 1], c=y_train, cmap="coolwarm", edgecolors="k", s=25)
plt.xlabel("Componente principal 1")
plt.ylabel("Componente principal 2")
plt.title("Frontera de decision de SVM (kernel RBF) - proyeccion PCA 2D")
plt.legend(handles=scatter.legend_elements()[0], labels=list(data.target_names))
plt.show()


## 7. Conclusiones

- SVM logra un **accuracy alto** en el dataset Breast Cancer, incluso con el kernel lineal más simple, lo que sugiere que las clases son bastante separables en este espacio de variables.
- El **kernel** y el hiperparámetro **C** son las decisiones de diseño más importantes: cambian directamente qué tan flexible es la frontera de decisión y cuántos vectores de soporte utiliza el modelo.
- Es fundamental **estandarizar las variables** antes de entrenar, dado que SVM se basa en distancias/márgenes entre puntos.
- La proyección en 2D (PCA) es solo una herramienta de visualización; el modelo real trabaja en el espacio completo de 30 variables, donde puede separar los datos de forma aún más precisa.
- Para un caso real (por ejemplo, en un entorno clínico), conviene además ajustar el umbral de decisión y los pesos de clase para minimizar específicamente los falsos negativos (tumores malignos clasificados como benignos), dado el alto costo de ese tipo de error.
